# Inference + TTA + Threshold Sweep — FADC-Bottleneck headline run

**Goal:** push the headline Dice past 0.70 using inference-time tricks on the existing `best_model.pth`. NO retraining.

**Stack applied on top of FADC-Bottleneck (training-val Dice 0.6851):**
- Sliding-window inference at `overlap=0.5` (training-time used 0.0 for speed) + Gaussian patch fusion
- 8-flip test-time augmentation (averages softmax probabilities across all 2³ spatial-axis flip combinations)
- Threshold sweep over `[0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]` — picks the one that maximizes mean val Dice

**Expected lift:** +0.015 to +0.035 stacked → 0.70–0.72 final.

**Compute:** Kaggle T4 x2, ~2.5–5 hours for 306 val cases × 8 TTA flips × overlap=0.5.

**Setup needed before launching:**
1. Upload `best_model.pth` from FADC-Bottleneck no-DS (training-val Dice 0.6851) as a new Kaggle dataset.
2. Attach that dataset + the existing 2ch preprocessed cache to this notebook.
3. Edit `CKPT_PATH` in the CONFIG cell below to point at the uploaded checkpoint.

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────

# Path to the uploaded best_model.pth Kaggle dataset.
# Example layout if your dataset slug is `bharathvemurik/fadc-bottleneck-best`
# and the file is at the root of the dataset:
CKPT_PATH = "/kaggle/input/fadc-bottleneck-best/best_model.pth"  # <-- EDIT THIS

MODEL_NAME = "unet3d_fadc_bottleneck"   # must match the checkpoint architecture

# 2ch preprocessed cache (the same one used for training)
DATA_ROOT              = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
PREPROCESSED_CACHE_DIR = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"

OUTPUT_DIR = "/kaggle/working/outputs/infer_tta_fadc_bottleneck"
CODE_DIR   = "/kaggle/working/FADC-3D"

# Inference knobs
OVERLAP        = 0.5                                       # 0.5 default; 0.0 = fastest, 0.5 = paper-grade
SW_BATCH_SIZE  = 4                                         # how many patches the sliding-window infers at once
THRESHOLDS     = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
USE_TTA        = True                                      # 8-flip TTA. Set False to skip TTA (1/8 the time).
LIMIT_N_CASES  = None                                      # set e.g. 5 for a quick smoke test before the full 306

In [ ]:
# ─────────────────────────────────────────────
# 1. INSTALL DEPENDENCIES
# ─────────────────────────────────────────────
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "monai",
    "--upgrade-strategy", "only-if-needed", "-q"
], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU. Enable T4 x2 in Settings -> Accelerator before running TTA inference.")

In [ ]:
# ─────────────────────────────────────────────
# 2. CLONE / UPDATE CODE FROM GITHUB
# ─────────────────────────────────────────────
import os

if os.path.exists(CODE_DIR):
    print("Repo already exists — pulling latest...")
    os.system(f"git -C {CODE_DIR} pull")
else:
    os.system(f"git clone https://github.com/Vemuri-BK/FADC-3D.git {CODE_DIR}")
    print("Repo cloned.")

sys.path.insert(0, CODE_DIR)
print(f"Code path: {CODE_DIR}")

In [ ]:
# ─────────────────────────────────────────────
# 3. VERIFY CHECKPOINT AND CACHE ARE MOUNTED
# ─────────────────────────────────────────────
from pathlib import Path

assert Path(CKPT_PATH).exists(), (
    f"Checkpoint not found: {CKPT_PATH}\n"
    f"Upload best_model.pth as a Kaggle dataset and attach it to this notebook, "
    f"then edit CKPT_PATH in the CONFIG cell."
)
print(f"Checkpoint OK : {CKPT_PATH}")
print(f"  Size        : {Path(CKPT_PATH).stat().st_size / 1e6:.1f} MB")

cache = Path(PREPROCESSED_CACHE_DIR)
assert cache.exists(), f"Cache not mounted: {cache}"
val_npzs = sorted((cache / "val").glob("*.npz"))
assert len(val_npzs) > 0, f"No .npz files in {cache / 'val'}"
print(f"Val cases     : {len(val_npzs)}")

# Quick peek at one .npz to confirm shape
import numpy as np
d = np.load(val_npzs[0])
print(f"Sample case   : {val_npzs[0].name}")
print(f"  image shape : {d['image'].shape}  (must start with 2 for 2ch)")
print(f"  label shape : {d['label'].shape}")
assert d['image'].shape[0] == 2, f"FATAL: expected 2-channel image, got {d['image'].shape[0]} channels"

In [ ]:
# ─────────────────────────────────────────────
# 4. QUICK MODEL SMOKE TEST (loads checkpoint, runs 1 forward pass)
# ─────────────────────────────────────────────
#   Catches a wrong --model / wrong-architecture checkpoint mismatch BEFORE
#   committing to the 3h inference run.
import torch, sys
sys.path.insert(0, CODE_DIR)
from infer_with_tta import build_model

device = torch.device("cuda")
ckpt   = torch.load(CKPT_PATH, map_location=device)
cfg    = ckpt["config"]
print(f"Checkpoint cfg model      : {ckpt.get('config', {}).get('model', '?')}")
print(f"Checkpoint best_dice      : {ckpt.get('best_dice', '?')}")
print(f"Checkpoint epoch          : {ckpt.get('epoch', '?')}")
print(f"Checkpoint patch_size     : {cfg['data']['patch_size']}")
print(f"Checkpoint deep_supervision: {cfg['model'].get('deep_supervision', False)}")

model = build_model(MODEL_NAME, cfg, device)
model.load_state_dict(ckpt["model"])
model.eval()

with torch.no_grad():
    x = torch.randn(1, cfg["model"]["in_channels"],
                    cfg["data"]["patch_size"][0],
                    cfg["data"]["patch_size"][1],
                    cfg["data"]["patch_size"][2]).to(device)
    y = model(x)
    assert y.shape[1] == cfg["model"]["out_channels"], f"unexpected output shape: {y.shape}"
print(f"Model forward OK. Output shape: {y.shape}")
del model, x, y, ckpt; torch.cuda.empty_cache()

In [ ]:
# ─────────────────────────────────────────────
# 5. RUN TTA INFERENCE
# ─────────────────────────────────────────────
#   Calls infer_with_tta.py as a subprocess so stdout streams cleanly to the
#   notebook output without buffering issues.
import os, subprocess, sys
os.makedirs(OUTPUT_DIR, exist_ok=True)

script = os.path.join(CODE_DIR, "infer_with_tta.py")

cmd = [
    sys.executable, "-u", script,
    "--ckpt",                    CKPT_PATH,
    "--model",                   MODEL_NAME,
    "--data_root",               DATA_ROOT,
    "--preprocessed_cache_dir",  PREPROCESSED_CACHE_DIR,
    "--output_dir",              OUTPUT_DIR,
    "--overlap",                 str(OVERLAP),
    "--sw_batch_size",           str(SW_BATCH_SIZE),
    "--thresholds",              *[str(t) for t in THRESHOLDS],
]
if not USE_TTA:
    cmd.append("--no_tta")
if LIMIT_N_CASES is not None:
    cmd += ["--limit", str(LIMIT_N_CASES)]

print("Command:")
print("  " + " ".join(cmd))
print("=" * 70)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = process.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
process.wait()
print(f"\nExit code: {process.returncode}")

In [ ]:
# ─────────────────────────────────────────────
# 6. LOAD AND DISPLAY RESULTS
# ─────────────────────────────────────────────
import json, os

res_path = os.path.join(OUTPUT_DIR, "tta_results.json")
if not os.path.exists(res_path):
    print(f"No results JSON found at {res_path} — check the cell above for errors.")
else:
    with open(res_path) as f:
        res = json.load(f)

    print("=" * 72)
    print(f"Model            : {res['model']}")
    print(f"Checkpoint       : {res['ckpt']}")
    print(f"Training-val Dice: {res.get('train_val_best_dice', '?')}")
    print(f"Overlap          : {res['overlap']}")
    print(f"TTA              : {'8-flip' if res['tta'] else 'off'}")
    print(f"Cases evaluated  : {res['n_cases']}")
    print(f"Wall time        : {res['total_minutes']:.1f} min")
    print("-" * 72)
    print(f"{'Threshold':<12}{'Dice':<12}{'IoU':<12}{'Sensitivity':<14}")
    for t_str, m in sorted(res["per_threshold"].items()):
        marker = " <-- BEST" if abs(float(t_str) - res["best_threshold"]) < 1e-6 else ""
        print(f"{t_str:<12}{m['dice_mean']:<12.4f}{m['iou_mean']:<12.4f}{m['sens_mean']:<14.4f}{marker}")
    print("-" * 72)
    print(f"BEST   Dice : {res['best_dice']:.4f}  @ threshold {res['best_threshold']:.2f}")
    print(f"       IoU  : {res['best_iou']:.4f}")
    print(f"       Sens : {res['best_sens']:.4f}")
    if res.get("train_val_best_dice") is not None:
        delta = res["best_dice"] - res["train_val_best_dice"]
        print(f"Lift vs training-val   : {delta:+.4f}")
    print("=" * 72)

    if res["best_dice"] >= 0.70:
        print("\nDice >= 0.70 — Phase 2 (Federated Learning) gate cleared.")
    else:
        gap = 0.70 - res["best_dice"]
        print(f"\nDice = {res['best_dice']:.4f} (gap to 0.70: {gap:+.4f}).")
        print("Fallback: 2-seed ensemble — rerun Bottleneck with --seed 42, average logits at inference. Adds ~7h Kaggle GPU.")

In [ ]:
# ─────────────────────────────────────────────
# 7. DOWNLOAD RESULTS BEFORE SESSION CLOSES
# ─────────────────────────────────────────────
#   /kaggle/working is wiped on session close — download tta_results.json now.
import os
from IPython.display import FileLink, display

res_path = os.path.join(OUTPUT_DIR, "tta_results.json")
if os.path.exists(res_path):
    print("Click the link to download tta_results.json:")
    display(FileLink(res_path))
else:
    print("tta_results.json not found — check the inference cell.")